In [1]:
import pandas as pd
import altair as alt
import os
import re
import numpy as np

analysis_date = "260408"

In [ ]:
# CSV file available upon request
df = pd.read_csv("data/raw_respondent_data.csv")
df = df[df['Finished'] == 'True']

condition_name_mapping = {
    'treatment_recommendation': 'Recommended Treatment',
    'best_options': 'Options to Consider',
    'worst_options': 'Options to Avoid',
}

df['AIInterface'] = df['AIInterface'].replace(condition_name_mapping)
df

In [3]:
df['Q17.1'].value_counts()

Q17.1
Yes, I'll review two additional cases    143
No, take me to the final questions        84
Name: count, dtype: int64

## Process Respondent-Level Data

In [4]:
# Clean demographic columns
df['practice_setting'] = df['practice_setting'].replace({
    'Academic, university aﬃliated': 'Academic, university affiliated',
    'Academic, not university aﬃliated': 'Academic, not university affiliated'
})
df['specialty'] = df['specialty'].where(~df['specialty'].str.startswith('Other'), 'Other')
df['clinical_time'] = df['clinical_time'].replace({
    'More than 50% but less than 95%': 'Majority',
    'Less than 50%': 'Minority',
    'All or almost all (95% or more)': 'Full',
})
df['Duration (in seconds)'] = df['Duration (in seconds)'].astype(float)

In [5]:
perception_cols = ['Productivity',
    'Effectiveness',
    'EaseOfUse',
    'HighOutputQuality',
    'WouldUse']
df = df.rename(columns={
    'Q3.2': 'Experience',
    'Q3.3': 'SepsisExpertise',
    'Q18.2_1': 'Productivity',
    'Q18.2_2': 'Effectiveness',
    'Q18.2_3': 'EaseOfUse',
    'Q18.2_4': 'HighOutputQuality',
    'Q18.2_5': 'WouldUse',
    'Q18.4': 'UsefulnessFreeText',
    'Q18.5': 'IdealAIRoleFreeText',
    'Q18.6': 'FeedbackFreeText',
})
df[perception_cols] = df[perception_cols].replace({
    'Strongly disagree': 1, 'Disagree': 2, 'Somewhat disagree': 3, 'Neither agree nor disagree': 4,
    'Somewhat agree': 5, 'Agree': 6, 'Strongly agree': 7
})
    
df['Total Satisfaction'] = df[perception_cols].sum(axis=1)

/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/295012698.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[perception_cols] = df[perception_cols].replace({


In [6]:
print(df['specialty'].value_counts())
print(df['specialty_10_TEXT'].value_counts())
print(df['clinical_time'].value_counts())
print(df['practice_setting'].value_counts())
print(df['Experience'].value_counts())
print(df['SepsisExpertise'].value_counts())

specialty
Internal medicine, pulmonary        160
Internal medicine, not pulmonary     16
Anesthesiology                       15
Emergency medicine                   14
Other                                14
Surgery                               8
Neurology                             1
Name: count, dtype: int64
specialty_10_TEXT
Critical Care Medicine                         1
pulmonary and critical care fellow             1
Emergency and internal medicine                1
Internal Medicine-Pediatrics                   1
Pulmonary/Critical Care                        1
Pulmonary Critical Care                        1
Critical Care                                  1
Ob/gyn                                         1
critical care medicine/ infectious diseases    1
Internal Medicine, Critical Care               1
Critical care and nephrology                   1
Name: count, dtype: int64
clinical_time
Minority    87
Majority    76
Full        62
Name: count, dtype: int64
practice_setting

In [7]:
# Clean multiple choice demographics
df['Experience'] = df['Experience'].replace({
    '< 1 year': 1,
    '1-2 years': 2,
    '3-5 years': 3,
    '5-10 years': 4,
    '> 10 years': 5
})
df['SepsisExpertise'] = df['SepsisExpertise'].replace({
    'Novice - no experience': 1,
    'Beginner - some working knowledge': 2,
    'Competent - good working and background knowledge': 3,
    'Proficient - deep understanding': 4,
    'Expert - authoritative knowledge': 5
})
print(df['Experience'].value_counts())
print(df['SepsisExpertise'].value_counts())

Experience
5    178
4     39
3     10
1      1
Name: count, dtype: int64
SepsisExpertise
4    105
5    105
3     18
Name: count, dtype: int64


/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/1565172145.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Experience'] = df['Experience'].replace({
/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/1565172145.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['SepsisExpertise'] = df['SepsisExpertise'].replace({


In [8]:
# exclude_flags = (df['clinical_time'] == 'None') | (df['Duration (min)'] <= 5)
# print(exclude_flags.sum(), "to exclude")
# df = df[~exclude_flags]

In [9]:
df.columns.tolist()

['StartDate',
 'EndDate',
 'Status',
 'IPAddress',
 'Progress',
 'Duration (in seconds)',
 'Finished',
 'RecordedDate',
 'ResponseId',
 'RecipientLastName',
 'RecipientFirstName',
 'RecipientEmail',
 'ExternalReference',
 'LocationLatitude',
 'LocationLongitude',
 'DistributionChannel',
 'UserLanguage',
 'Q_RecaptchaScore',
 'Q_DuplicateRespondent',
 'Q_BallotBoxStuffing',
 'Experience',
 'SepsisExpertise',
 'Q5.1_First Click',
 'Q5.1_Last Click',
 'Q5.1_Page Submit',
 'Q5.1_Click Count',
 'Q5.3',
 'Q5.4',
 'Q5.5',
 'Q6.1_First Click',
 'Q6.1_Last Click',
 'Q6.1_Page Submit',
 'Q6.1_Click Count',
 'Q6.3',
 'Q6.4',
 'Q6.5',
 'Q6.7',
 'Q6.8',
 'Q6.8_8_TEXT',
 'Q7.1_First Click',
 'Q7.1_Last Click',
 'Q7.1_Page Submit',
 'Q7.1_Click Count',
 'Q7.3',
 'Q7.4',
 'Q7.5',
 'Q8.1_First Click',
 'Q8.1_Last Click',
 'Q8.1_Page Submit',
 'Q8.1_Click Count',
 'Q8.3',
 'Q8.4',
 'Q8.5',
 'Q8.7',
 'Q8.8',
 'Q8.8_8_TEXT',
 'Q9.1_First Click',
 'Q9.1_Last Click',
 'Q9.1_Page Submit',
 'Q9.1_Click Count'

In [10]:
no_influences = df[[c for c in df.columns if c.endswith(".8")]].apply(lambda x: x.str.contains("did not affect my thinking at all", case=False), axis=0)
df['SelfReportedUnaffected'] = no_influences.sum(axis=1) >= (~pd.isna(no_influences)).sum(axis=1) - 1
df['SelfReportedAffected'] = no_influences.sum(axis=1) == 0
df['SelfReportedUnaffected'].mean(), df['SelfReportedAffected'].mean()

(0.23684210526315788, 0.4780701754385965)

In [11]:
df.assign(Condition=df['AIInterface'])[[
    'ResponseId',
    'AIInterface',
    'specialty',
    'specialty_10_TEXT',
    'practice_setting',
    'clinical_time',
    'gender',
    'age',
    'race',
    'hispanic',
    'specialty_7_TEXT',
    'ethnicity',
    'Duration (in seconds)',
    'Experience',
    'SepsisExpertise',
    'Productivity',
    'Effectiveness',
    'EaseOfUse',
    'HighOutputQuality',
    'WouldUse',
    'Total Satisfaction',
    'SelfReportedUnaffected',
    'SelfReportedAffected',
    'UsefulnessFreeText',
    'IdealAIRoleFreeText',
    'FeedbackFreeText']].to_csv(f"data/cleaned_respondent_level_{analysis_date}.csv", index=False)

## Decision-Level Data

In [12]:
df.columns = [re.sub(r"^Q(5|6|7|8|9|10|11|12|13|14|15|16)\.(.*)$", r"Decision_\2_\1", c) for c in df.columns]
long_df = pd.wide_to_long(
    df,
    ['Decision_1_First Click_', 'Decision_1_Last Click_', 'Decision_1_Page Submit_', 'Decision_1_Click Count_',
         'Decision_3_', 'Decision_4_', 'Decision_5_', 'Decision_7_', 'Decision_8_', 'Decision_8_8_TEXT_'],
    'ResponseId',
    'Case' 
).reset_index()
long_df.head(10)

,ResponseId,Case,AIInterface,DeborahNum,DeborahShown,DistributionChannel,Duration (in seconds),EaseOfUse,Effectiveness,Email,...,Decision_1_First Click_,Decision_1_Last Click_,Decision_1_Page Submit_,Decision_1_Click Count_,Decision_3_,Decision_4_,Decision_5_,Decision_7_,Decision_8_,Decision_8_8_TEXT_
0,R_6RxfhYGfh8jlhJg,5,Recommended Treatment,0,1,email,773.0,5,4,tsuangw@ccf.org,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,R_5jUO17A3YDcW9sl,5,Options to Consider,5,1,email,831.0,3,2,rishikvashisht@yahoo.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,R_6P5X1V9iJ8kyAxx,5,Options to Avoid,0,1,email,154.0,5,5,rezaimd@yahoo.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,R_6dpSgwpnoJJbFiV,5,Options to Consider,2,1,email,246.0,6,6,Arubinmd@gmail.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R_5KcNS6HUy5bmkq6,5,Options to Avoid,4,1,email,940.0,4,3,harold.palevsky@uphs.upenn.edu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,R_1lJRXsqOpUaonTM,5,Recommended Treatment,3,1,email,553.0,7,6,Kachen@mdanderson.org,...,87.773,98.572,98.575,6,Give up to 1 L of IV fluid.,Wean the existing vasopressor.,Somewhat confident,NaN,NaN,NaN
6,R_6si6i6VtEOCwBC1,5,Options to Consider,1,1,email,1189.0,4,3,Bauer.Philippe@mayo.edu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,R_6GwYOIVLUFVNQuD,5,Recommended Treatment,0,1,email,857.0,2,2,jlberkowitz@gmail.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,R_1Y7wZBGuUgNF7I5,5,Options to Avoid,5,1,email,936.0,4,2,herman.dyal@gmail.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,R_1DU7Rz35vHWdkLj,5,Options to Avoid,2,1,email,816.0,6,1,gajic.ognjen@mayo.edu,...,95.716,104.711,104.716,4,Give neither IV fluids nor diuretics.,Continue the existing vasopressor.,Somewhat confident,NaN,NaN,NaN


In [13]:
influences = long_df['Decision_8_'].str.split(",").explode().unique()
for c in influences:
    if pd.isna(c): continue
    long_df[f'Influence: {c}'] = long_df['Decision_8_'].str.contains(c).fillna(False)

long_df['AIInterface'] = long_df['AIInterface'].where(long_df['Case'] % 2 == 0, 'No AI')
long_df['Case'] = ((long_df['Case'] - 5) // 2).replace({
    0: 'Deborah',
    1: 'Heather',
    2: 'Esther',
    3: 'Richard',
    4: 'Margaret',
    5: 'Mike'
})
long_df

/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/2071281912.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_df[f'Influence: {c}'] = long_df['Decision_8_'].str.contains(c).fillna(False)
/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/2071281912.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_df[f'Influence: {c}'] = long_df['Decision_8_'].str.contains(c).fillna(False)
/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/2071281912.py:4: FutureWarning: Downcasting object dtype arrays on .fillna,

,ResponseId,Case,AIInterface,DeborahNum,DeborahShown,DistributionChannel,Duration (in seconds),EaseOfUse,Effectiveness,Email,...,Influence: Prompted me to assess the patient further,Influence: Prompted me to consider an alternative plan,Influence: I'm not sure,Influence: Helped me decide between alternatives,Influence: Made me more confident in my plan,Influence: Affected my thinking in some other way (please describe):,Influence: Made me second-guess my plan,Influence: Did not affect my thinking at all,Influence: Gave me an initial plan,Influence: Helped me form an initial assessment of the patient
0,R_6RxfhYGfh8jlhJg,Deborah,No AI,0,1,email,773.0,5,4,tsuangw@ccf.org,...,False,False,False,False,False,False,False,False,False,False
1,R_5jUO17A3YDcW9sl,Deborah,No AI,5,1,email,831.0,3,2,rishikvashisht@yahoo.com,...,False,False,False,False,False,False,False,False,False,False
2,R_6P5X1V9iJ8kyAxx,Deborah,No AI,0,1,email,154.0,5,5,rezaimd@yahoo.com,...,False,False,False,False,False,False,False,False,False,False
3,R_6dpSgwpnoJJbFiV,Deborah,No AI,2,1,email,246.0,6,6,Arubinmd@gmail.com,...,False,False,False,False,False,False,False,False,False,False
4,R_5KcNS6HUy5bmkq6,Deborah,No AI,4,1,email,940.0,4,3,harold.palevsky@uphs.upenn.edu,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,R_1LbANtwLORsGJH3,Mike,Options to Avoid,2,1,email,1325.0,7,2,NaN,...,False,False,False,False,False,False,False,False,False,False
2732,R_7o6mhqmhzTcqepy,Mike,Recommended Treatment,1,1,email,832.0,4,3,NaN,...,False,False,False,False,False,False,False,False,False,False
2733,R_32bZLFt0SUnMy4y,Mike,Options to Consider,3,1,email,2063.0,6,4,NaN,...,False,False,False,False,False,False,True,False,False,False
2734,R_3zj2hLcEzVQzRV3,Mike,Options to Consider,NaN,NaN,email,1061.0,6,6,NaN,...,False,False,False,True,False,False,False,False,False,False


In [14]:
nums = []
choice_change_fluids = []
choice_change_vaso = []
for i, row in long_df.iterrows():
    nums.append(row[row['Case'] + 'Num'])
    choice_change_fluids.append(row[f"__js_ChoiceChange{row['Case']}Fluids"])
    choice_change_vaso.append(row[f"__js_ChoiceChange{row['Case']}Vaso"])
    
long_df = long_df.assign(CaseNum=nums, ChoiceChangeFluids=choice_change_fluids, ChoiceChangeVaso=choice_change_vaso)
long_df

,ResponseId,Case,AIInterface,DeborahNum,DeborahShown,DistributionChannel,Duration (in seconds),EaseOfUse,Effectiveness,Email,...,Influence: Helped me decide between alternatives,Influence: Made me more confident in my plan,Influence: Affected my thinking in some other way (please describe):,Influence: Made me second-guess my plan,Influence: Did not affect my thinking at all,Influence: Gave me an initial plan,Influence: Helped me form an initial assessment of the patient,CaseNum,ChoiceChangeFluids,ChoiceChangeVaso
0,R_6RxfhYGfh8jlhJg,Deborah,No AI,0,1,email,773.0,5,4,tsuangw@ccf.org,...,False,False,False,False,False,False,False,0,1,1
1,R_5jUO17A3YDcW9sl,Deborah,No AI,5,1,email,831.0,3,2,rishikvashisht@yahoo.com,...,False,False,False,False,False,False,False,5,1,2
2,R_6P5X1V9iJ8kyAxx,Deborah,No AI,0,1,email,154.0,5,5,rezaimd@yahoo.com,...,False,False,False,False,False,False,False,0,1,1
3,R_6dpSgwpnoJJbFiV,Deborah,No AI,2,1,email,246.0,6,6,Arubinmd@gmail.com,...,False,False,False,False,False,False,False,2,1,NaN
4,R_5KcNS6HUy5bmkq6,Deborah,No AI,4,1,email,940.0,4,3,harold.palevsky@uphs.upenn.edu,...,False,False,False,False,False,False,False,4,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,R_1LbANtwLORsGJH3,Mike,Options to Avoid,2,1,email,1325.0,7,2,NaN,...,False,False,False,False,False,False,False,NaN,NaN,NaN
2732,R_7o6mhqmhzTcqepy,Mike,Recommended Treatment,1,1,email,832.0,4,3,NaN,...,False,False,False,False,False,False,False,NaN,NaN,NaN
2733,R_32bZLFt0SUnMy4y,Mike,Options to Consider,3,1,email,2063.0,6,4,NaN,...,False,False,False,True,False,False,False,0,1,1
2734,R_3zj2hLcEzVQzRV3,Mike,Options to Consider,NaN,NaN,email,1061.0,6,6,NaN,...,True,False,False,False,False,False,False,1,2,1


In [15]:
long_df = long_df[[
    'ResponseId', 'Case', 'AIInterface', 'CaseNum', 'Decision_1_First Click_',
       'Decision_1_Last Click_', 'Decision_1_Page Submit_',
       'Decision_1_Click Count_', 'Decision_3_', 'Decision_4_', 'Decision_5_',
       'Decision_7_', 'Decision_8_8_TEXT_', 'ChoiceChangeFluids', 'ChoiceChangeVaso'
] + [c for c in long_df.columns if c.startswith("Influence:")]]
long_df = long_df[~pd.isna(long_df['Decision_3_'])]

In [16]:
alt.Chart(long_df).mark_bar().encode(
    alt.X("Decision_1_Page Submit_:Q", bin=alt.Bin(extent=[0, 600], steps=[5, 10, 15])),
    alt.Y("count():Q")
)

alt.Chart(...)

In [18]:
long_df = long_df.rename(columns={
    "Q3.2": "Experience",
    "Q3.3": "SepsisExpertise",
    "Decision_1_First Click_": "First Click Time",
    "Decision_1_Last Click_": "Last Click Time",
    "Decision_1_Page Submit_": "Page Submit Time",
    "Decision_1_Click Count_": "Click Count",
    "Decision_3_": "Fluid",
    "Decision_4_": "Vasopressor",
    "Decision_5_": "Confidence",
    "Decision_7_": "AIQuality",
    "Decision_8_8_TEXT_": "Other Influences",
})

## Add Decision Quality

In [19]:
fluid_terms = {
    'Give neither IV fluids nor diuretics.': 0,
    "Stop diuretics and don't give IV fluids.": 0,
    'Give up to 1 L of IV fluid.': 1,
    'Stop diuretics and give up to 1 L of IV fluid.': 1,
    'Give more than 1 L of IV fluid.': 2,
    'Stop diuretics and give more than 1 L of IV fluid.': 2,
    'Give a diuretic.': 3,
    'Continue the diuretic.': 3
}

vaso_terms = {
    'Do not give vasopressors.': 0,
    'Wean both existing vasopressors.': 0,
    'Wean the existing vasopressor.': 0,
    'Continue the existing vasopressor.': 1,
    'Start a vasopressor.': 1,
    'Wean the secondary vasopressor.': 1, 
    'Add another vasopressor.': 2,
    'Continue both vasopressors.': 2,
    'Start multiple vasopressors.': 2,
    'Wean the existing vasopressor ONLY IF I ordered fluid and there is a positive response in less than four hours.': 4,
    'Wean one or both vasopressors ONLY IF I ordered fluid and there is a positive response in less than four hours.': 4,
    'Start a vasopressor ONLY IF I ordered fluid and there is NO positive response in less than four hours.': 3,
    'Add another vasopressor ONLY IF I ordered fluid and there is NO positive response in less than four hours.': 5,
    'Wean the secondary vasopressor now, and wean the other vasopressor ONLY IF I ordered fluid and there is a positive response in less than four hours.': 5,
}
print(set(long_df['Fluid'].unique()) - set(fluid_terms.keys()))
print(set(long_df['Vasopressor'].unique()) - set(vaso_terms.keys()))

fluid_nums = long_df['Fluid'].replace(fluid_terms)
vaso_nums = long_df['Vasopressor'].replace(vaso_terms)

invalid_decision = (vaso_nums >= 3) & ~fluid_nums.isin([1, 2])
print("invalid decision:", invalid_decision.sum())

long_df = long_df[~invalid_decision]
fluid_nums = fluid_nums[~invalid_decision]
vaso_nums = vaso_nums[~invalid_decision]

base_decision_numbers = fluid_nums * 3 + vaso_nums
base_decision_numbers = base_decision_numbers.where(vaso_nums < 3,
                                                    12 + (fluid_nums - 1) + (vaso_nums - 3))
base_decision_numbers = base_decision_numbers.where(vaso_nums < 4,
                                                    12 + (fluid_nums - 1) * 2 + (vaso_nums - 4))
base_decision_numbers += 1
base_decision_numbers.value_counts()

set()
set()
invalid decision: 19


/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/3751363438.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fluid_nums = long_df['Fluid'].replace(fluid_terms)
/var/folders/gj/cy6x5zxs74j__yh6fwqs72500000gp/T/ipykernel_34144/3751363438.py:32: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vaso_nums = long_df['Vasopressor'].replace(vaso_terms)


2     398
13    178
11    143
5     135
14     74
3      66
1      48
8      33
12     28
10     21
4      15
15     11
16     10
6       8
9       6
7       5
Name: count, dtype: int64

In [20]:
decision_num_df = pd.concat([long_df, fluid_nums.rename('fluid_num'), vaso_nums.rename('vaso_num'), base_decision_numbers.rename('decision_num')], axis=1)
for num in sorted(decision_num_df['decision_num'].unique()):
    print(num)
    print(decision_num_df[decision_num_df['decision_num'] == num][['Fluid', 'Vasopressor', 'fluid_num', 'vaso_num']].value_counts())

1
Fluid                                     Vasopressor                       fluid_num  vaso_num
Give neither IV fluids nor diuretics.     Do not give vasopressors.         0          0           21
                                          Wean the existing vasopressor.    0          0           16
Stop diuretics and don't give IV fluids.  Do not give vasopressors.         0          0            7
Give neither IV fluids nor diuretics.     Wean both existing vasopressors.  0          0            4
Name: count, dtype: int64
2
Fluid                                     Vasopressor                         fluid_num  vaso_num
Give neither IV fluids nor diuretics.     Start a vasopressor.                0          1           243
Stop diuretics and don't give IV fluids.  Start a vasopressor.                0          1            70
Give neither IV fluids nor diuretics.     Continue the existing vasopressor.  0          1            48
                                          Wean the se

In [21]:
quality_df = pd.read_csv("../../expert_panel/quality_ratings.csv").rename(columns={
    'Final': 'QualityScore',
    'E1': 'QualityScore_E1',
    'E2': 'QualityScore_E2',
    'E3': 'QualityScore_E3',
    'E4': 'QualityScore_E4',
    'E5': 'QualityScore_E5',
})

In [22]:
scored_df = pd.merge(decision_num_df, quality_df, how='left', left_on=['Case', 'decision_num'], right_on=['Case', 'Decision']).drop(columns=['Decision'])
scored_df

,ResponseId,Case,AIInterface,CaseNum,First Click Time,Last Click Time,Page Submit Time,Click Count,Fluid,Vasopressor,...,Influence: Helped me form an initial assessment of the patient,fluid_num,vaso_num,decision_num,QualityScore_E1,QualityScore_E2,QualityScore_E3,QualityScore_E4,QualityScore_E5,QualityScore
0,R_1lJRXsqOpUaonTM,Deborah,No AI,3,87.773,98.572,98.575,6,Give up to 1 L of IV fluid.,Wean the existing vasopressor.,...,False,1,0,4,2,4,4,5,3,4
1,R_1DU7Rz35vHWdkLj,Deborah,No AI,2,95.716,104.711,104.716,4,Give neither IV fluids nor diuretics.,Continue the existing vasopressor.,...,False,0,1,2,4,7,3,2,7,4
2,R_6q509aZS2abtXbd,Deborah,No AI,3,79.675,87.921,87.928,4,Give up to 1 L of IV fluid.,Add another vasopressor ONLY IF I ordered flui...,...,False,1,5,14,6,6,4,3,4,4
3,R_6Wrg2o8DQo2FsjL,Deborah,No AI,0,23.235,43.681,43.694,4,Give up to 1 L of IV fluid.,Wean the existing vasopressor.,...,False,1,0,4,2,4,4,5,3,4
4,R_65Yj36eVkr1xSKZ,Deborah,No AI,1,22.559,108.769,108.784,15,Give up to 1 L of IV fluid.,Add another vasopressor ONLY IF I ordered flui...,...,False,1,5,14,6,6,4,3,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,R_3gZieJvrRzDGHCN,Mike,Options to Consider,1,168.603,180.971,180.979,4,Give neither IV fluids nor diuretics.,Start a vasopressor.,...,True,0,1,2,5,6,6,6,6,6
1175,R_7SdWLi0O3s3mFJz,Mike,Recommended Treatment,2,4.473,603.355,603.363,14,Give up to 1 L of IV fluid.,Start a vasopressor ONLY IF I ordered fluid an...,...,False,1,3,13,7,5,5,5,4,5
1176,R_32bZLFt0SUnMy4y,Mike,Options to Consider,0,39.706,131.113,131.131,5,Give up to 1 L of IV fluid.,Start a vasopressor.,...,False,1,1,5,7,6,5,6,5,6
1177,R_3zj2hLcEzVQzRV3,Mike,Options to Consider,1,27.718,182.456,182.476,8,Give a diuretic.,Start a vasopressor.,...,False,3,1,11,4,3,5,5,3,4


In [25]:
scored_df['First Click Time'] = scored_df['First Click Time'].astype(float)
scored_df['Last Click Time'] = scored_df['Last Click Time'].astype(float)
scored_df['Page Submit Time'] = scored_df['Page Submit Time'].astype(float)

scored_df['TimeSpent'] = scored_df['Page Submit Time']
scored_df['LongTime'] = scored_df['Page Submit Time'] >= 120 # scored_df['Page Submit Time'].median()

scored_df['InfluenceGroupAssessment'] = scored_df[['Influence: Prompted me to assess the patient further',
                                              'Influence: Helped me form an initial assessment of the patient']].any(axis=1)
scored_df['InfluenceGroupPlansConsidered'] = scored_df[['Influence: Gave me an initial plan',
                                                   'Influence: Prompted me to consider an alternative plan']].any(axis=1)
scored_df['InfluenceGroupPlansEvaluated'] = scored_df[['Influence: Helped me decide between alternatives']].any(axis=1)
scored_df['InfluenceGroupConfidence'] = scored_df[['Influence: Made me more confident in my plan',
                                                    'Influence: Made me second-guess my plan']].any(axis=1)
influence_columns = ['Influence: Prompted me to assess the patient further',
 'Influence: Prompted me to consider an alternative plan',
 "Influence: I'm not sure",
 'Influence: Made me more confident in my plan',
 'Influence: Affected my thinking in some other way (please describe):',
 'Influence: Made me second-guess my plan',
 'Influence: Did not affect my thinking at all',
 'Influence: Gave me an initial plan',
 'Influence: Helped me decide between alternatives',
 'Influence: Helped me form an initial assessment of the patient']
short_names = [
    'InfluenceAssessFurther',
    'InfluenceConsiderAlternative',
    "InfluenceImNotSure",
    'InfluenceMoreConfident',
    'InfluenceOther',
    'InfluenceSecondGuess',
    'InfluenceNoEffect',
    'InfluenceInitialPlan',
    'InfluenceHelpDecide',
    'InfluenceInitialAssessment'
]
scored_df = scored_df.rename(columns=dict(zip(influence_columns, short_names)))

In [27]:
scored_df.to_csv(f"data/cleaned_decision_level_{analysis_date}.csv", index=False)

## AI Agreement Data

In [145]:
# Look at the agreement with the various AI interface outputs
ai_quality = pd.read_csv('../../expert_panel/AI_plans_labeled_separate.csv').rename(columns={
    'patient_name': 'Case',
    'ai_condition': 'AI Label'
})
ai_quality['AI Label'] = ai_quality['AI Label'].replace(condition_name_mapping)

In [147]:
# Define pairs of decisions considered "similar" for measuring partial concordance.
# Partial concordance means either the fluid or the vasopressor decision is identical
# to the AI, and the other decision is "similar" to the AI.
vaso_similar_decision = {
    "Deborah": { (0, 1), (0, 4), (2, 5), (1, 2) },
    "Heather": { (0, 1), (1, 3), (1, 2), (0, 3), },
    "Esther": { (0, 1), (1, 3), (1, 2), (0, 3) },
    "Richard": { (0, 1), (1, 2), (1, 4), (0, 4), (1, 5), (0, 5), },
    "Margaret": { (0, 1), (1, 3), (1, 2), (0, 3) },
    "Mike": { (0, 1), (1, 3), (1, 2), (0, 3) }
}

fluid_similar_decision = {
    "Deborah": { (0, 1), (0, 3), (1, 2) },
    "Heather": { (0, 1), (0, 3), (1, 2) },
    "Esther": { (0, 1), (0, 3), (1, 2) },
    "Richard": { (0, 1), (0, 3), (1, 2) },
    "Margaret": { (0, 1), (0, 3), (1, 2) },
    "Mike": { (0, 1), (0, 3), (1, 2) },
}

concordances = []
for i, row in scored_df.iterrows():
    case_name = row["Case"]
    ai_labels = [row['AIInterface']] if row['AIInterface'] != 'No AI' else scored_df['AIInterface'].unique()
    for ai in ai_labels:
        if (ai_quality['AI Label'] == ai).sum() == 0: continue
        ai_fluid_nums = ai_quality[(ai_quality['AI Label'] == ai) & (ai_quality['Case'] == case_name)]['fluid_num'].values.tolist()
        ai_vaso_nums = ai_quality[(ai_quality['AI Label'] == ai) & (ai_quality['Case'] == case_name)]['vaso_num'].values.tolist()
        ai_decision_nums = ai_quality[(ai_quality['AI Label'] == ai) & (ai_quality['Case'] == case_name)]['decision_num'].values.tolist()
        ai_quality_scores = ai_quality[(ai_quality['AI Label'] == ai) & (ai_quality['Case'] == case_name)]['quality'].values.tolist()
        tx_rec_scores = ai_quality[(ai_quality['AI Label'] == 'Recommended Treatment') & (ai_quality['Case'] == case_name)]['quality'].values.tolist()

        if ai == 'Options to Avoid':
            concord = row['decision_num'] not in ai_decision_nums
            partial_concord = any(
                (row['fluid_num'] == fluid_num
                    and row['vaso_num'] != vaso_num
                    ) or
                (row['vaso_num'] == vaso_num
                    and row['fluid_num'] != fluid_num
                    )
                for fluid_num, vaso_num in zip(ai_fluid_nums, ai_vaso_nums)
            )
            
            concordance_type = ('X' if row['decision_num'] in ai_decision_nums else ('Partial' if partial_concord else 'Full'))
        else:
            concord = row['decision_num'] in ai_decision_nums
            partial_concord = any(
                (row['fluid_num'] == fluid_num
                    and (row['vaso_num'] == vaso_num or
                        (vaso_num, row['vaso_num']) in vaso_similar_decision[case_name] or
                            (row['vaso_num'], vaso_num) in vaso_similar_decision[case_name])
                    ) or
                (row['vaso_num'] == vaso_num
                    and (row['fluid_num'] == fluid_num or
                        (fluid_num, row['fluid_num']) in fluid_similar_decision[case_name] or
                            (row['fluid_num'], fluid_num) in fluid_similar_decision[case_name])
                    )
                for fluid_num, vaso_num in zip(ai_fluid_nums, ai_vaso_nums)
            )
            
            concordance_type = ('Full' if concord else ('Partial' if partial_concord else 'X'))
        
        result = {
            **row,
            'HasAI': row["AIInterface"] != 'No AI',
            'AgreementWith': ai,
            'FullConcordance': concord, # concordance_type == 'Full',
            'PartialConcordance': partial_concord, # concordance_type == 'Partial',
            'ConcordanceType': concordance_type,
            'BlindAgreement': row['AIInterface'] != 'No AI' and concord and row['Page Submit Time'] <= 120,
            'SimulatedQualityScore': row['SimulatedQualityScore']
        }
        concordances.append(result)

ai_concordance_df = pd.DataFrame(concordances)


In [148]:
column_order = [
    'ResponseId',
    'Case',
    'AgreementWith',
    'HasAI',
    'AIInterface',
    'FullConcordance',
    'PartialConcordance',
    'ConcordanceType'
]
column_order += [c for c in ai_concordance_df.columns if c not in column_order]

for ai in ai_concordance_df['AgreementWith'].unique():
    ai_concordance_df[ai_concordance_df['AgreementWith'] == ai][column_order].to_csv(f"data/agreement_with_{ai}_{analysis_date}.csv", index=False)
    
ai_concordance_df[ai_concordance_df['HasAI'] == True][column_order].to_csv(f"data/agreement_all_ai_{analysis_date}.csv", index=False)

In [108]:
# Get individual dfs with an overall "agreer" score
for condition in ai_concordance_df['AgreementWith'].unique():
    agreer_flags = ai_concordance_df[ai_concordance_df['AIInterface'] == condition].groupby('ResponseId').agg({'FullConcordance': lambda g: g.sum() >= 0.5}).rename(columns={'FullConcordance': 'HighConcordance'}).reset_index()
    # agreer_flags['HighConcordance'] = agreer_flags['HighConcordance'] > 0.5 # agreer_flags['HighConcordance'].median() - 1e-3
    condition_agreement = pd.merge(agreer_flags, ai_concordance_df.drop_duplicates(['ResponseId', 'Case']), how='left', on='ResponseId')
    print(condition_agreement[['HighConcordance', 'HasAI']].value_counts())
    condition_agreement[condition_agreement['AIInterface'] == 'No AI'][['ResponseId', 'HighConcordance'] + column_order[1:]].to_csv(f"data/agreement_no_ai_{condition}_251204.csv", index=False)

HighConcordance  HasAI
True             True     240
False            True      55
True             False     53
False            False     15
Name: count, dtype: int64
HighConcordance  HasAI
True             True     300
                 False     70
False            True      30
                 False     10
Name: count, dtype: int64
HighConcordance  HasAI
True             True     329
                 False     77
Name: count, dtype: int64


In [109]:
len(condition_agreement['ResponseId'].unique())

79

In [149]:
import re
df = pd.read_csv("data/cleaned_respondent_level_251204.csv")

print(len(df['UsefulnessFreeText'][df['UsefulnessFreeText'].fillna('') != '']), "out of", len(df))
useful_responses = df['UsefulnessFreeText'][df['UsefulnessFreeText'].fillna('') != '']
lengths = useful_responses.apply(lambda r: len(re.split(r'\W+', r)))
print(lengths.median(), lengths.min(), lengths.max())
for ai in df['AIInterface'].unique():
    print(ai)
    print("USEFULNESS\n\n")
    print('\n\n'.join(df[df['AIInterface'] == ai]['UsefulnessFreeText'][df['UsefulnessFreeText'].fillna('') != ''].values.tolist()))
    print("\n\n")
    print("IDEAL ROLE\n\n")
    print('\n\n'.join(df[df['AIInterface'] == ai]['IdealAIRoleFreeText'][df['IdealAIRoleFreeText'].fillna('') != ''].values.tolist()))
    print("\n\n")
    print("MISC\n\n")
    print('\n\n'.join(df[df['AIInterface'] == ai]['FeedbackFreeText'][df['FeedbackFreeText'].fillna('') != ''].values.tolist()))
    print("\n\n")

146 out of 228
25.0 3 291
Recommended Treatment
USEFULNESS


simple direct suggestions

Recommendations were counter to what I have generally been taught regarding sepsis care

Firstly, sorry if I didn't answer the questions correctly about how Sepsis AI influenced me. I said it provided an initial plan largely of I agreed with the plan. I felt that it should be clearer whether or not the AI would re-evaluate the offered plan. The plan did not always seem sufficiently comprehensive. I generally felt more confident in my plan than the AI's plan. 

Simplified clinical decisions

useful as it framed the issue, but not useful as some recommendations were not appropriate

Confirmed my plan.

It would help if sepsis AI gave some form of a confidence rating. For example, in the one example, the cirrhotic patient had AKI and one might start vasopressors for concern for HRS but it said no vasopressors. It it had a low confidence on that answer versus a high confidence, one might look at that ad